# Transformer From Scratch

**Task:** Character-level language model trained on TinyShakespeare  
**Architecture:** GPT-style decoder-only Transformer

##  Table of Contents
1. [Setup & GPU Check](#setup)
2. [Tokenizer (Character-level)](#tokenizer)
3. [Dataset & DataLoader](#dataset)
4. [Model Architecture](#architecture)
   - 4a. [Token + Positional Embeddings](#embeddings)
   - 4b. [Multi-Head Self-Attention (Q/K/V)](#attention)
   - 4c. [MLP / Feed-Forward Block](#mlp)
   - 4d. [Transformer Block (Residual + LayerNorm)](#block)
   - 4e. [Full GPT Model](#gpt)
5. [Training Script](#training)
6. [Generation](#generation)
7. [Inspection Utilities (for future experiments)](#inspection)
8. [Save / Load Checkpoint](#checkpoint)

---
## 1. Setup & GPU Check <a id='setup'></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import time
import os
import urllib.request
from dataclasses import dataclass
from typing import Optional, Tuple

# ── GPU Check ──────────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    # Enable TF32 for faster matmuls on Ampere+ (T4 supports it). TF32 (TensorFloat-32) is a specialized 19-bit numerical format used by NVIDIA Tensor Cores for AI training and inference, providing up to 10x faster performance than standard 32-bit floating-point (FP32) while maintaining comparable accuracy. It keeps the 8-bit dynamic range of FP32 for stability but uses the 10-bit precision of FP16.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print(' No GPU found — Using CPU for training')

torch.manual_seed(42)
print('\nSetup complete')

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB

Setup complete


---
## 2. Hyperparameters & Config

In [2]:
@dataclass
class GPTConfig:
    # ── Architecture ───────────────────────────────────────────────────────
    vocab_size   : int   = 65        # will be updated after tokenizer is built
    block_size   : int   = 256       # context length (sequence length)
    n_layer      : int   = 6         # number of transformer blocks
    n_head       : int   = 6         # number of attention heads
    n_embd       : int   = 384       # embedding dimension (d_model)
    dropout      : float = 0.1
    bias         : bool  = False     # bias in Linear layers & LayerNorm

    # ── Training ───────────────────────────────────────────────────────────
    batch_size   : int   = 64        # sequences per batch
    max_iters    : int   = 5000      # total training steps
    eval_interval: int   = 500       # evaluate every N steps
    eval_iters   : int   = 200       # batches to average for eval loss
    log_interval : int   = 100       # print loss every N steps

    # ── Optimizer ──────────────────────────────────────────────────────────
    learning_rate: float = 3e-4
    weight_decay : float = 0.1
    beta1        : float = 0.9
    beta2        : float = 0.95
    grad_clip    : float = 1.0

    # ── LR Schedule ────────────────────────────────────────────────────────
    warmup_iters : int   = 100
    lr_decay_iters: int  = 5000      # = max_iters (cosine decay to min_lr)
    min_lr       : float = 3e-5      # = learning_rate / 10

    @property
    def head_size(self):
        assert self.n_embd % self.n_head == 0
        return self.n_embd // self.n_head

cfg = GPTConfig()
print(cfg)

GPTConfig(vocab_size=65, block_size=256, n_layer=6, n_head=6, n_embd=384, dropout=0.1, bias=False, batch_size=64, max_iters=5000, eval_interval=500, eval_iters=200, log_interval=100, learning_rate=0.0003, weight_decay=0.1, beta1=0.9, beta2=0.95, grad_clip=1.0, warmup_iters=100, lr_decay_iters=5000, min_lr=3e-05)


---
## 3. Tokenizer (Character-level) <a id='tokenizer'></a>

In [3]:
# ── Download TinyShakespeare ────────────────────────────────────────────────
DATA_URL  = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
DATA_PATH = 'input.txt'

if not os.path.exists(DATA_PATH):
    print('Downloading TinyShakespeare...')
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print('Done.')

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    text = f.read()

print(f'Dataset length : {len(text):,} characters')
print(f'Sample         : {repr(text[:80])}')

# ── Build vocabulary ────────────────────────────────────────────────────────
chars    = sorted(set(text))
vocab_size = len(chars)
print(f'\nVocab size     : {vocab_size}')
print(f'Vocab          : {repr("".join(chars))}')

# char ↔ int mappings
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# ── Sanity check ────────────────────────────────────────────────────────────
sample = 'Hello, World!'
print(f'\nEncode test: {encode(sample)}')
print(f'Decode test: {decode(encode(sample))}')

# Update config
cfg.vocab_size = vocab_size

Done.
Dataset length : 1,115,394 characters
Sample         : 'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.'

Vocab size     : 65
Vocab          : "\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"

Encode test: [20, 43, 50, 50, 53, 6, 1, 35, 53, 56, 50, 42, 2]
Decode test: Hello, World!


---
## 4. Dataset & DataLoader <a id='dataset'></a>

In [4]:
# ── Encode full dataset ─────────────────────────────────────────────────────
data = torch.tensor(encode(text), dtype=torch.long)
print(f'Full data tensor: {data.shape}  dtype={data.dtype}')

# ── Train / Val split (90 / 10) ─────────────────────────────────────────────
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]
print(f'Train tokens    : {len(train_data):,}')
print(f'Val tokens      : {len(val_data):,}')

# ── Batch sampler ──────────────────────────────────────────────────────────
def get_batch(split: str, cfg: GPTConfig):
    """
    Returns (x, y) where:
      x : (batch_size, block_size)  — input token ids
      y : (batch_size, block_size)  — target token ids (x shifted by 1)
    """
    data_split = train_data if split == 'train' else val_data
    # Random start indices
    ix = torch.randint(len(data_split) - cfg.block_size, (cfg.batch_size,))
    x  = torch.stack([data_split[i   : i + cfg.block_size] for i in ix])
    y  = torch.stack([data_split[i+1 : i + cfg.block_size + 1] for i in ix])
    return x.to(device), y.to(device)

# ── Smoke test ──────────────────────────────────────────────────────────────
xb, yb = get_batch('train', cfg)
print(f'\nBatch x: {xb.shape}   y: {yb.shape}')
print(f'x[0,:8] = {xb[0,:8].tolist()}')
print(f'y[0,:8] = {yb[0,:8].tolist()}  (x shifted left by 1)')

Full data tensor: torch.Size([1115394])  dtype=torch.int64
Train tokens    : 1,003,854
Val tokens      : 111,540

Batch x: torch.Size([64, 256])   y: torch.Size([64, 256])
x[0,:8] = [54, 43, 63, 1, 58, 46, 43, 0]
y[0,:8] = [43, 63, 1, 58, 46, 43, 0, 19]  (x shifted left by 1)


---
## 5. Model Architecture <a id='architecture'></a>

### Pipeline
```
tokens
  │
  ▼
[Token Embedding]  +  [Positional Embedding]
  │
  ▼
[Dropout]
  │
  ▼  ┌─────────────────────────────────────────┐
     │  Transformer Block  (×N)                │
     │                                         │
     │  x = x + MultiHeadAttention(LN(x))      │
     │  x = x + MLP(LN(x))                     │
     └─────────────────────────────────────────┘
  │
  ▼
[LayerNorm]
  │
  ▼
[Linear head → vocab logits]
  │
  ▼
Cross-Entropy Loss
```

### 5a. Multi-Head Self-Attention <a id='attention'></a>

**Math:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**Shapes** (for one head):
- Q, K, V : `(B, T, head_size)`
- scores   : `(B, T, T)` — every token attends to every prior token
- output   : `(B, T, head_size)`

In [5]:
class MultiHeadAttention(nn.Module):
    """
    Multi-head causal self-attention.

    All heads are computed in parallel via a single (3 * n_embd) projection
    then split — same math, faster than a loop over heads.

    Key tensors to inspect for KV-cache experiments:
      self.k_cache, self.v_cache  (set them from outside during inference)
    """

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head    = cfg.n_head
        self.n_embd    = cfg.n_embd
        self.head_size = cfg.n_embd // cfg.n_head
        self.dropout_p = cfg.dropout

        # ── Projections ──────────────────────────────────────────────────
        # Single matrix projects x → Q, K, V concatenated
        # Shape: (n_embd, 3 * n_embd)
        self.c_attn = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=cfg.bias)
        # Output projection
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=cfg.bias)

        # ── Regularization ───────────────────────────────────────────────
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)

        # ── Causal mask ──────────────────────────────────────────────────
        # Lower-triangular matrix of 1s: token i can attend to tokens 0..i
        # Registered as buffer so it moves to GPU with .to(device)
        self.register_buffer(
            'mask',
            torch.tril(torch.ones(cfg.block_size, cfg.block_size))
            .view(1, 1, cfg.block_size, cfg.block_size)   # (1,1,T,T)
        )

        # ── KV cache placeholders (used during inference) ─────────────────
        # Will be populated externally for KV-cache experiments
        self.k_cache = None
        self.v_cache = None

    def forward(
        self,
        x: torch.Tensor,                        # (B, T, C)  C = n_embd
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Tuple]]:

        B, T, C = x.shape

        # ── 1. Compute Q, K, V ───────────────────────────────────────────
        # Project x → (B, T, 3*C), then split into Q K V each (B, T, C)
        qkv = self.c_attn(x)                           # (B, T, 3*C)
        q, k, v = qkv.split(self.n_embd, dim=2)        # each (B, T, C)

        # ── 2. Reshape for multi-head ────────────────────────────────────
        # (B, T, C) → (B, n_head, T, head_size)
        def split_heads(t):
            return t.view(B, T, self.n_head, self.head_size).transpose(1, 2)

        q = split_heads(q)   # (B, nh, T, hs)
        k = split_heads(k)   # (B, nh, T, hs)
        v = split_heads(v)   # (B, nh, T, hs)

        # ── KV cache append (for inference experiments) ──────────────────
        if use_cache:
            if self.k_cache is not None:
                k = torch.cat([self.k_cache, k], dim=2)  # grow along T
                v = torch.cat([self.v_cache, v], dim=2)
            self.k_cache = k
            self.v_cache = v

        T_k = k.shape[2]  # key sequence length (may differ from T with cache)

        # ── 3. Scaled dot-product attention ──────────────────────────────
        # scores: (B, nh, T, T_k)
        scale  = 1.0 / math.sqrt(self.head_size)
        scores = (q @ k.transpose(-2, -1)) * scale      # (B, nh, T, T_k)

        # ── 4. Causal mask (future tokens → -inf before softmax) ─────────
        scores = scores.masked_fill(
            self.mask[:, :, :T, :T_k] == 0,
            float('-inf')
        )

        # ── 5. Softmax → attention weights ───────────────────────────────
        weights = F.softmax(scores, dim=-1)             # (B, nh, T, T_k)
        weights = self.attn_dropout(weights)

        # ── 6. Weighted sum of values ─────────────────────────────────────
        out = weights @ v                               # (B, nh, T, hs)

        # ── 7. Merge heads back ───────────────────────────────────────────
        # (B, nh, T, hs) → (B, T, nh*hs) = (B, T, C)
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        # ── 8. Output projection ─────────────────────────────────────────
        out = self.resid_dropout(self.c_proj(out))      # (B, T, C)

        return out, (k, v) if use_cache else None


# ── Quick shape test ────────────────────────────────────────────────────────
attn_test = MultiHeadAttention(cfg).to(device)
x_test    = torch.randn(2, cfg.block_size, cfg.n_embd, device=device)
out_test, _ = attn_test(x_test)
print(f'Attention output shape: {out_test.shape}')  # expect (2, 256, 384)
del attn_test, x_test, out_test

Attention output shape: torch.Size([2, 256, 384])


### 5b. MLP / Feed-Forward Block <a id='mlp'></a>

Each token independently passes through:
$$\text{MLP}(x) = W_2 \cdot \text{GELU}(W_1 x + b_1) + b_2$$

The hidden dimension is **4× the embedding dimension** (standard GPT convention).

In [6]:
class MLP(nn.Module):
    """
    Position-wise feed-forward network.

    x (B, T, C)  →  linear(C → 4C)  →  GELU  →  linear(4C → C)  →  dropout
    """

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=cfg.bias),
            nn.GELU(),                                  # smooth non-linearity
            nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=cfg.bias),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)   # (B, T, C) → (B, T, C)

### 5c. Transformer Block (Pre-LN + Residual) <a id='block'></a>

**Pre-norm formulation** (more stable than original "post-norm"):
```
x = x + Attention(LayerNorm(x))   # residual stream 1
x = x + MLP(LayerNorm(x))         # residual stream 2
```

In [7]:
class TransformerBlock(nn.Module):
    """
    One full transformer layer:
      Pre-LayerNorm → MultiHeadAttention → residual
      Pre-LayerNorm → MLP               → residual
    """

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.ln1  = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.attn = MultiHeadAttention(cfg)
        self.ln2  = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.mlp  = MLP(cfg)

    def forward(
        self,
        x: torch.Tensor,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Tuple]]:

        # ── Attention sublayer ─────────────────────────────────────────────
        # LayerNorm BEFORE attention (pre-LN)
        attn_out, kv = self.attn(self.ln1(x), use_cache=use_cache)
        x = x + attn_out                          # residual connection

        # ── MLP sublayer ───────────────────────────────────────────────────
        x = x + self.mlp(self.ln2(x))             # residual connection

        return x, kv

### 5d. Full GPT Model <a id='gpt'></a>

In [9]:
class GPT(nn.Module):
    """
    Decoder-only transformer (GPT style).

    Forward pass shapes:
      idx    : (B, T)  — token ids
      logits : (B, T, vocab_size)
      loss   : scalar cross-entropy (or None during inference)
    """

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg

        # ── Embedding tables ──────────────────────────────────────────────
        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb   = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.emb_drop  = nn.Dropout(cfg.dropout)

        # ── Transformer blocks ────────────────────────────────────────────
        self.blocks = nn.ModuleList([
            TransformerBlock(cfg) for _ in range(cfg.n_layer)
        ])

        # ── Final layer norm & language model head ────────────────────────
        self.ln_f  = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)

        # ── Weight tying ──────────────────────────────────────────────────
        # Token embedding and lm_head share weights (saves params, improves perf)
        self.token_emb.weight = self.lm_head.weight

        # ── Init weights ──────────────────────────────────────────────────
        self.apply(self._init_weights)
        # Scale residual projections by 1/sqrt(2*n_layer) as in GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(
        self,
        idx: torch.Tensor,          # (B, T)
        targets: Optional[torch.Tensor] = None,  # (B, T)
        use_cache: bool = False,
    ):
        B, T = idx.shape
        assert T <= self.cfg.block_size, \
            f'Sequence length {T} > block_size {self.cfg.block_size}'

        # ── 1. Embeddings ─────────────────────────────────────────────────
        pos = torch.arange(T, device=idx.device)     # (T,)

        tok_emb = self.token_emb(idx)                # (B, T, C)
        pos_emb = self.pos_emb(pos)                  # (T, C) → broadcast

        # tok_emb + pos_emb: where in vocab AND where in sequence
        x = self.emb_drop(tok_emb + pos_emb)        # (B, T, C)

        # ── 2. Transformer blocks ─────────────────────────────────────────
        kv_cache = []
        for block in self.blocks:
            x, kv = block(x, use_cache=use_cache)
            kv_cache.append(kv)

        # ── 3. Final LayerNorm ────────────────────────────────────────────
        x = self.ln_f(x)                             # (B, T, C)

        # ── 4. Language model head ────────────────────────────────────────
        logits = self.lm_head(x)                     # (B, T, vocab_size)

        # ── 5. Loss (only during training) ───────────────────────────────
        loss = None
        if targets is not None:
            # Flatten: (B*T, vocab_size) vs (B*T,)
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
            )

        return logits, loss, kv_cache

    # ── Parameter count ────────────────────────────────────────────────────
    def num_parameters(self, non_embedding=True):
        n = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n -= self.pos_emb.weight.numel()
        return n

    # ── Optimizer factory ──────────────────────────────────────────────────
    def configure_optimizer(self, cfg: GPTConfig):
        """
        AdamW with weight decay on 2D params only (weights, not biases/LN).
        This mirrors the GPT-3 / nanoGPT setup.
        """
        decay_params    = [p for n, p in self.named_parameters()
                           if p.dim() >= 2 and p.requires_grad]
        no_decay_params = [p for n, p in self.named_parameters()
                           if p.dim() < 2  and p.requires_grad]

        groups = [
            {'params': decay_params,    'weight_decay': cfg.weight_decay},
            {'params': no_decay_params, 'weight_decay': 0.0},
        ]
        return torch.optim.AdamW(
            groups,
            lr=cfg.learning_rate,
            betas=(cfg.beta1, cfg.beta2),
            fused=True if device == 'cuda' else False,  # faster on GPU
        )


# ── Instantiate & inspect ────────────────────────────────────────────────────
model = GPT(cfg).to(device)
total_params = model.num_parameters(non_embedding=False)
non_emb_params = model.num_parameters(non_embedding=True)

print(f'Model architecture:')
print(f'  n_layer    = {cfg.n_layer}')
print(f'  n_head     = {cfg.n_head}')
print(f'  n_embd     = {cfg.n_embd}')
print(f'  block_size = {cfg.block_size}')
print(f'  vocab_size = {cfg.vocab_size}')
print(f'\nTotal parameters     : {total_params:,}')
print(f'Non-embedding params : {non_emb_params:,}')

# Forward pass test
xb, yb  = get_batch('train', cfg)
logits, loss, _ = model(xb, yb)
print(f'\nForward pass OK')
print(f'  logits shape : {logits.shape}')  # (B, T, vocab_size)
print(f'  initial loss : {loss.item():.4f}  (expect ≈ ln({cfg.vocab_size}) = {math.log(cfg.vocab_size):.2f})')

Model architecture:
  n_layer    = 6
  n_head     = 6
  n_embd     = 384
  block_size = 256
  vocab_size = 65

Total parameters     : 10,745,088
Non-embedding params : 10,646,784

Forward pass OK
  logits shape : torch.Size([64, 256, 65])
  initial loss : 4.2360  (expect ≈ ln(65) = 4.17)


---
## 6. Training Script <a id='training'></a>

In [10]:
def get_lr(it: int, cfg: GPTConfig) -> float:
    """
    Cosine decay learning rate schedule with linear warmup.

    Phase 1: linear warmup from 0 → lr over warmup_iters steps
    Phase 2: cosine decay from lr → min_lr over lr_decay_iters steps
    Phase 3: constant at min_lr
    """
    # Warmup
    if it < cfg.warmup_iters:
        return cfg.learning_rate * it / cfg.warmup_iters
    # Beyond decay horizon
    if it > cfg.lr_decay_iters:
        return cfg.min_lr
    # Cosine decay
    decay_ratio = (it - cfg.warmup_iters) / (cfg.lr_decay_iters - cfg.warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return cfg.min_lr + coeff * (cfg.learning_rate - cfg.min_lr)

In [11]:
@torch.no_grad()
def estimate_loss(model: GPT, cfg: GPTConfig) -> dict:
    """Average loss over eval_iters batches for train and val splits."""
    model.eval()
    losses = {}
    for split in ('train', 'val'):
        split_losses = torch.zeros(cfg.eval_iters)
        for k in range(cfg.eval_iters):
            x, y = get_batch(split, cfg)
            _, loss, _ = model(x, y)
            split_losses[k] = loss.item()
        losses[split] = split_losses.mean().item()
    model.train()
    return losses

In [ ]:
# ── Re-init fresh model ──────────────────────────────────────────────────────
torch.manual_seed(42)
model = GPT(cfg).to(device)
optimizer = model.configure_optimizer(cfg)

# Use automatic mixed precision (float16) on CUDA for ~2× throughput
scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))
ctx    = torch.autocast(device_type=device, dtype=torch.float16) \
         if device == 'cuda' else torch.no_grad.__class__  # dummy ctx on CPU

# ── Training history ────────────────────────────────────────────────────────
train_losses = []
val_losses   = []
lrs          = []

print(f'Starting training for {cfg.max_iters} iterations...')
print(f'Device: {device}  |  Mixed precision: {device == "cuda"}')
print('-' * 70)

t0 = time.time()

for step in range(cfg.max_iters):

    # ── LR schedule ───────────────────────────────────────────────────────
    lr = get_lr(step, cfg)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    lrs.append(lr)

    # ── Periodic evaluation ───────────────────────────────────────────────
    if step % cfg.eval_interval == 0 or step == cfg.max_iters - 1:
        losses = estimate_loss(model, cfg)
        train_losses.append((step, losses['train']))
        val_losses.append((step, losses['val']))
        elapsed = time.time() - t0
        print(f'step {step:5d} | train loss {losses["train"]:.4f} '
              f'| val loss {losses["val"]:.4f} '
              f'| lr {lr:.2e} '
              f'| {elapsed:.1f}s elapsed')

    # ── Forward pass ──────────────────────────────────────────────────────
    xb, yb = get_batch('train', cfg)

    if device == 'cuda':
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            _, loss, _ = model(xb, yb)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
    else:
        _, loss, _ = model(xb, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()

    optimizer.zero_grad(set_to_none=True)

    # ── Logging ───────────────────────────────────────────────────────────
    if step % cfg.log_interval == 0:
        tokens_per_sec = (
            cfg.batch_size * cfg.block_size * cfg.log_interval
            / max(time.time() - t0, 1e-8)
        )

total_time = time.time() - t0
print('-' * 70)
print(f'Training complete in {total_time/60:.1f} min')

/tmp/ipykernel_12008/2750072582.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))


Starting training for 5000 iterations...
Device: cuda  |  Mixed precision: True
----------------------------------------------------------------------
step     0 | train loss 4.2160 | val loss 4.2157 | lr 0.00e+00 | 54.4s elapsed


In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Loss curves
    ax = axes[0]
    t_steps, t_vals = zip(*train_losses)
    v_steps, v_vals = zip(*val_losses)
    ax.plot(t_steps, t_vals, label='Train', linewidth=2)
    ax.plot(v_steps, v_vals, label='Val',   linewidth=2, linestyle='--')
    ax.set_xlabel('Step'); ax.set_ylabel('Loss'); ax.set_title('Training Loss')
    ax.legend(); ax.grid(True, alpha=0.3)

    # LR schedule
    ax = axes[1]
    ax.plot(lrs, linewidth=2, color='orange')
    ax.set_xlabel('Step'); ax.set_ylabel('Learning Rate'); ax.set_title('LR Schedule (Warmup + Cosine Decay)')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Plot saved to training_curves.png')
except ImportError:
    print('matplotlib not available — skip plot')

---
## 7. Generation <a id='generation'></a>

In [ ]:
@torch.no_grad()
def generate(
    model: GPT,
    idx: torch.Tensor,          # (B, T) seed token ids
    max_new_tokens: int,
    temperature: float = 1.0,   # <1.0 = sharper, >1.0 = more random
    top_k: Optional[int] = None,  # keep only top-k logits before sampling
) -> torch.Tensor:
    """
    Autoregressive generation (no KV cache — straightforward).
    For KV-cache version, see the Inspection section below.
    """
    model.eval()
    for _ in range(max_new_tokens):
        # Crop context to block_size
        idx_cond = idx[:, -model.cfg.block_size:]

        # Forward — only need logits of last token
        logits, _, _ = model(idx_cond)
        logits = logits[:, -1, :]                  # (B, vocab_size)

        # Temperature scaling
        logits = logits / temperature

        # Top-k filtering
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')

        # Sample from distribution
        probs = F.softmax(logits, dim=-1)          # (B, vocab_size)
        idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
        idx = torch.cat([idx, idx_next], dim=1)    # (B, T+1)

    return idx


# ── Generate sample text ─────────────────────────────────────────────────────
print('=== Generated text (temperature=0.8, top_k=40) ===')
seed = torch.zeros((1, 1), dtype=torch.long, device=device)  # start with '\n'
out  = generate(model, seed, max_new_tokens=500, temperature=0.8, top_k=40)
print(decode(out[0].tolist()))
print('===')

---
## 8. Inspection Utilities <a id='inspection'></a>
> These are designed to help you experiment with **KV cache**, **continuous batching**, and **speculative decoding**.

In [ ]:
# ── 8a. Visualize attention patterns ────────────────────────────────────────

@torch.no_grad()
def get_attention_maps(model: GPT, text_prompt: str):
    """
    Returns attention weight tensors for all layers and heads.
    Shape per layer: (n_head, T, T)
    """
    model.eval()
    tokens = torch.tensor(encode(text_prompt), dtype=torch.long).unsqueeze(0).to(device)
    T = tokens.shape[1]

    attn_maps = {}

    # Hook to grab attention weights
    hooks = []
    for layer_idx, block in enumerate(model.blocks):
        def make_hook(idx):
            def hook_fn(module, inp, out):
                # Re-compute weights from Q, K (they're not stored by default)
                pass  # we'll use the manual approach below
            return hook_fn
        hooks.append(block.attn.register_forward_hook(make_hook(layer_idx)))

    # Manual extraction
    x = model.emb_drop(model.token_emb(tokens) + model.pos_emb(torch.arange(T, device=device)))

    for layer_idx, block in enumerate(model.blocks):
        normed = block.ln1(x)
        attn   = block.attn
        B, T_curr, C = normed.shape

        qkv   = attn.c_attn(normed)
        q, k, v = qkv.split(attn.n_embd, dim=2)
        nh, hs  = attn.n_head, attn.head_size

        q = q.view(B, T_curr, nh, hs).transpose(1, 2)
        k = k.view(B, T_curr, nh, hs).transpose(1, 2)
        v = v.view(B, T_curr, nh, hs).transpose(1, 2)

        scale   = 1.0 / math.sqrt(hs)
        scores  = (q @ k.transpose(-2, -1)) * scale
        scores  = scores.masked_fill(attn.mask[:,:,:T_curr,:T_curr] == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)       # (B, nh, T, T)
        attn_maps[f'layer_{layer_idx}'] = weights.squeeze(0).cpu()  # (nh, T, T)

        # Continue forward
        out = weights @ v
        out = out.transpose(1, 2).contiguous().view(B, T_curr, C)
        out = attn.resid_dropout(attn.c_proj(out))
        x   = x + out
        x   = x + block.mlp(block.ln2(x))

    for h in hooks:
        h.remove()

    return attn_maps, encode(text_prompt)


def plot_attention(attn_maps, token_ids, layer=0, head=0):
    try:
        import matplotlib.pyplot as plt
        chars = [itos[t] for t in token_ids]
        w     = attn_maps[f'layer_{layer}'][head].numpy()

        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(w, cmap='Blues', vmin=0)
        ax.set_xticks(range(len(chars))); ax.set_xticklabels(chars, fontsize=9)
        ax.set_yticks(range(len(chars))); ax.set_yticklabels(chars, fontsize=9)
        ax.set_title(f'Attention weights — Layer {layer}, Head {head}')
        plt.colorbar(im, ax=ax)
        plt.tight_layout()
        plt.show()
    except ImportError:
        print('matplotlib not available')


# ── Example ──────────────────────────────────────────────────────────────────
prompt      = 'To be or not to be'
attn_maps, tids = get_attention_maps(model, prompt)
print(f'Attention maps extracted for prompt: "{prompt}"')
print(f'Layers: {list(attn_maps.keys())}')
print(f'Shape per layer: {attn_maps["layer_0"].shape}  (n_head, T, T)')

plot_attention(attn_maps, tids, layer=0, head=0)

In [ ]:
# ── 8b. KV Cache — naive implementation stub ─────────────────────────────────
"""
HOW KV CACHE WORKS
──────────────────
Normal autoregressive generation recomputes K, V for ALL past tokens
at every step — O(T²) total work.

With KV cache:
  • Prefill step : full forward pass → store K, V for each layer
  • Decode steps : forward only the NEW token, append its K/V to cache
  → O(T) per step instead of O(T²)

The MultiHeadAttention class above already has:
  self.k_cache  and  self.v_cache

You just need to:
  1. Call model(prompt_tokens) to prefill
  2. For each new token:
       - Pass only the single new token
       - Each block's attention will append to the cache
"""

@torch.no_grad()
def generate_with_kv_cache(
    model: GPT,
    prompt: str,
    max_new_tokens: int,
    temperature: float = 0.8,
    top_k: int = 40,
) -> str:
    """
    Generation with KV cache.
    Prefill: process prompt in one shot.
    Decode: one token at a time, reusing cached K, V.
    """
    model.eval()

    # ── Clear existing caches ─────────────────────────────────────────────
    for block in model.blocks:
        block.attn.k_cache = None
        block.attn.v_cache = None

    # ── Prefill ───────────────────────────────────────────────────────────
    tokens   = encode(prompt)
    idx      = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)
    logits, _, _ = model(idx, use_cache=True)       # populates k_cache, v_cache
    # Take last token's logit
    logits   = logits[:, -1, :] / temperature
    if top_k:
        v, _ = torch.topk(logits, top_k)
        logits[logits < v[:, [-1]]] = float('-inf')
    probs    = F.softmax(logits, dim=-1)
    next_tok = torch.multinomial(probs, 1)          # (1, 1)
    generated = [next_tok.item()]

    # ── Decode (one token at a time) ──────────────────────────────────────
    for _ in range(max_new_tokens - 1):
        logits, _, _ = model(next_tok, use_cache=True)  # single token
        logits  = logits[:, -1, :] / temperature
        if top_k:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float('-inf')
        probs    = F.softmax(logits, dim=-1)
        next_tok = torch.multinomial(probs, 1)
        generated.append(next_tok.item())

    # ── Clear caches ──────────────────────────────────────────────────────
    for block in model.blocks:
        block.attn.k_cache = None
        block.attn.v_cache = None

    return prompt + decode(generated)


# ── Compare speed: without vs with KV cache ──────────────────────────────────
N_GEN   = 200
seed_p  = '\n'

# Without cache
t0 = time.time()
seed_t = torch.zeros((1,1), dtype=torch.long, device=device)
out_nc = generate(model, seed_t, N_GEN, temperature=0.8, top_k=40)
t_no_cache = time.time() - t0

# With cache
t0 = time.time()
out_wc = generate_with_kv_cache(model, seed_p, N_GEN, temperature=0.8, top_k=40)
t_with_cache = time.time() - t0

print(f'Generating {N_GEN} tokens:')
print(f'  Without KV cache : {t_no_cache:.2f}s  ({N_GEN/t_no_cache:.1f} tok/s)')
print(f'  With KV cache    : {t_with_cache:.2f}s  ({N_GEN/t_with_cache:.1f} tok/s)')
print(f'  Speedup          : {t_no_cache/t_with_cache:.1f}×')

In [ ]:
# ── 8c. Speculative Decoding — skeleton ──────────────────────────────────────
"""
SPECULATIVE DECODING
────────────────────
Idea: use a small draft model to propose K tokens in one shot,
      then verify them in ONE parallel pass of the large (target) model.

Steps:
  1. Draft model auto-regressively generates k tokens: t1, t2, ..., tk
  2. Target model runs forward pass on (context + t1..tk) in PARALLEL
  3. Accept/reject each draft token using rejection sampling:
       - If target_prob(ti) >= draft_prob(ti): accept
       - Else: accept with prob target/draft, reject otherwise
  4. On first rejection at position j, resample from corrected distribution
  5. Repeat from accepted position

In this single-model setup, you can simulate it by:
  - Using a smaller config (fewer layers) as the "draft" model
  - Using the full model as the "target"
"""

def speculative_decode_step(
    draft_model: GPT,
    target_model: GPT,
    context: torch.Tensor,   # (1, T)
    k: int = 4,              # speculation length
    temperature: float = 1.0,
) -> Tuple[torch.Tensor, int]:
    """
    One speculative decoding step.
    Returns: (accepted_tokens_tensor, n_accepted)
    """
    draft_model.eval(); target_model.eval()

    with torch.no_grad():
        # ── 1. Draft: generate k candidate tokens ─────────────────────────
        draft_tokens = []
        draft_probs  = []
        ctx = context.clone()

        for _ in range(k):
            logits, _, _ = draft_model(ctx[:, -draft_model.cfg.block_size:])
            logits = logits[:, -1, :] / temperature
            probs  = F.softmax(logits, dim=-1)          # (1, V)
            tok    = torch.multinomial(probs, 1)         # (1, 1)
            draft_tokens.append(tok)
            draft_probs.append(probs[0, tok[0, 0]].item())
            ctx = torch.cat([ctx, tok], dim=1)

        # ── 2. Target: single parallel forward on context + draft tokens ──
        target_input  = ctx                              # context + k drafts
        target_logits, _, _ = target_model(
            target_input[:, -target_model.cfg.block_size:]
        )
        # We need target probs at positions T, T+1, ..., T+k-1
        T_orig = context.shape[1]

        accepted_tokens = []
        n_accepted = 0

        # ── 3. Accept/Reject each draft token ─────────────────────────────
        for i, (tok, dp) in enumerate(zip(draft_tokens, draft_probs)):
            pos_in_target = T_orig - (target_input.shape[1] - target_logits.shape[1]) + i
            t_logits = target_logits[0, i, :] / temperature
            t_probs  = F.softmax(t_logits, dim=-1)
            tp       = t_probs[tok[0, 0]].item()

            accept_prob = min(1.0, tp / (dp + 1e-8))
            if torch.rand(1).item() < accept_prob:
                accepted_tokens.append(tok)
                n_accepted += 1
            else:
                # Rejection: sample from corrected distribution
                corrected = torch.clamp(t_probs - F.softmax(t_logits, dim=-1), min=0)
                corrected = corrected / corrected.sum()
                fallback  = torch.multinomial(corrected.unsqueeze(0), 1)
                accepted_tokens.append(fallback)
                break

        result = torch.cat(accepted_tokens, dim=1) if accepted_tokens else \
                 torch.empty((1, 0), dtype=torch.long, device=context.device)

    return result, n_accepted


print('Speculative decoding skeleton ready.')
print('To use: create a smaller GPTConfig (e.g. n_layer=2) → draft_model')
print('        and use the trained model as target_model.')

---
## 9. Save / Load Checkpoint <a id='checkpoint'></a>

In [ ]:
CKPT_PATH = 'transformer_ckpt.pt'

# ── Save ────────────────────────────────────────────────────────────────────
checkpoint = {
    'model_state'    : model.state_dict(),
    'optimizer_state': optimizer.state_dict(),
    'config'         : cfg,
    'train_losses'   : train_losses,
    'val_losses'     : val_losses,
    'vocab'          : {'stoi': stoi, 'itos': itos},
}
torch.save(checkpoint, CKPT_PATH)
print(f'Checkpoint saved → {CKPT_PATH}  ({os.path.getsize(CKPT_PATH)/1e6:.1f} MB)')

# ── Load ────────────────────────────────────────────────────────────────────
def load_checkpoint(path: str, device: str = 'cpu'):
    ckpt      = torch.load(path, map_location=device)
    cfg_load  = ckpt['config']
    model_new = GPT(cfg_load).to(device)
    model_new.load_state_dict(ckpt['model_state'])
    stoi_l    = ckpt['vocab']['stoi']
    itos_l    = ckpt['vocab']['itos']
    print(f'Loaded checkpoint from {path}')
    return model_new, cfg_load, stoi_l, itos_l

# Verify round-trip
model_loaded, _, _, _ = load_checkpoint(CKPT_PATH, device)
seed = torch.zeros((1, 1), dtype=torch.long, device=device)
out  = generate(model_loaded, seed, max_new_tokens=100, temperature=0.8)
print('\nSample from loaded model:')
print(decode(out[0].tolist()))